# Logistic Regression Evaluation — Fetal Health Classification

This notebook gives a fair Logistic Regression comparison against your teammate's KNN model.

**Important idea:** use the same dataset cleaning, the same train/test split, the same random seed, and comparable evaluation metrics.

Target classes:

- `1.0` = Normal
- `2.0` = Suspect
- `3.0` = Pathological

## 0. Imports

We use a pipeline so that scaling is fitted only on the training folds during cross-validation. This avoids data leakage.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import matplotlib.pyplot as plt

RANDOM_STATE = 42
TEST_SIZE = 0.20
CLASS_NAMES = ["Normal", "Suspect", "Pathological"]

## 1. Load the dataset

This notebook expects the dataset at `datasets/fetal_health.csv` inside your GitLab repository.

In [ ]:
possible_paths = [
    Path("datasets/fetal_health.csv"),
    Path("../datasets/fetal_health.csv"),
    Path("fetal_health.csv"),
    Path("data/fetal_health.csv"),
]

DATA_PATH = None
for path in possible_paths:
    if path.exists():
        DATA_PATH = path
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find fetal_health.csv. Expected it at datasets/fetal_health.csv. "
        "If your notebook is inside a notebooks folder, ../datasets/fetal_health.csv will also work."
    )

print("Using dataset:", DATA_PATH)

df = pd.read_csv(DATA_PATH)
print("Original shape:", df.shape)
df.head()


## 2. Basic data check and duplicate removal

We remove duplicate rows so the same observation does not appear in both train and test sets.

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("
Missing values per column:")
print(df.isna().sum())

n_duplicates = df.duplicated().sum()
print("
Duplicate rows:", n_duplicates)

df = df.drop_duplicates().copy()
print("Shape after removing duplicates:", df.shape)

print("
Class distribution:")
print(df["fetal_health"].value_counts().sort_index())

## 3. Separate features and target

`X` contains the input features. `y` contains the class label.

In [ ]:
X = df.drop("fetal_health", axis=1)
y = df["fetal_health"]

print("Number of samples:", X.shape[0])
print("Number of features:", X.shape[1])
print("Target classes:", sorted(y.unique()))

## 4. Train/test split

We use a stratified split because the classes are imbalanced. This keeps similar class proportions in training and test data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

split_distribution = pd.DataFrame({
    "train_count": y_train.value_counts().sort_index(),
    "test_count": y_test.value_counts().sort_index(),
})
split_distribution["train_percent"] = 100 * split_distribution["train_count"] / len(y_train)
split_distribution["test_percent"] = 100 * split_distribution["test_count"] / len(y_test)
split_distribution.round(2)

## 5. Logistic Regression pipeline

Logistic Regression needs feature scaling because it is affected by feature magnitudes.

The pipeline ensures this order:

1. Scale training data.
2. Train Logistic Regression.
3. Apply the same scaler to validation/test data.

In [ ]:
log_reg_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
])

## 6. Hyperparameter optimization with cross-validation

We tune `C` and `class_weight`.

- `C` controls regularization. Small `C` = stronger regularization. Large `C` = weaker regularization.
- `class_weight="balanced"` gives more weight to minority classes.

We optimize `f1_macro` because the dataset is imbalanced.

In [ ]:
param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__class_weight": [None, "balanced"],
    "classifier__solver": ["lbfgs"],
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

grid_search = GridSearchCV(
    estimator=log_reg_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=1,  # change to -1 for faster runs if your PC handles it well
    return_train_score=True,
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)
print("
Best cross-validation macro-F1:", round(grid_search.best_score_, 3))

## 7. Show cross-validation results

This table helps you explain why the chosen Logistic Regression setting was selected.

In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results = cv_results[[
    "param_classifier__C",
    "param_classifier__class_weight",
    "mean_train_score",
    "mean_test_score",
    "std_test_score",
    "rank_test_score",
]].sort_values("rank_test_score")

cv_results.round(3).head(10)

## 8. Final test evaluation

The test set is used only after hyperparameter tuning is finished.

In [ ]:
best_log_reg = grid_search.best_estimator_
y_pred = best_log_reg.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
balanced_acc = balanced_accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print("Logistic Regression Test Results")
print("--------------------------------")
print("Accuracy:", round(accuracy, 3))
print("Balanced Accuracy:", round(balanced_acc, 3))
print("Macro-F1:", round(macro_f1, 3))
print("Weighted-F1:", round(weighted_f1, 3))

print("
Classification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=CLASS_NAMES,
    digits=3,
))

## 9. Confusion matrix: absolute counts

Rows are true classes. Columns are predicted classes. The diagonal entries are correct predictions.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_NAMES,
)

disp.plot(values_format="d")
plt.title("Logistic Regression Confusion Matrix — Counts")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 10. Confusion matrix: row-normalized percentages

This version shows the percentage per true class. It is useful when classes are imbalanced.

In [ ]:
cm_normalized = confusion_matrix(y_test, y_pred, normalize="true")

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_normalized,
    display_labels=CLASS_NAMES,
)

disp.plot(values_format=".2f")
plt.title("Logistic Regression Confusion Matrix — Row Normalized")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 11. Save results for GitLab comparison

This creates small output files that your teammate can compare with KNN results.

In [ ]:
results_dir = Path("results")
figures_dir = Path("figures")
results_dir.mkdir(exist_ok=True)
figures_dir.mkdir(exist_ok=True)

log_reg_results = {
    "Model": "Logistic Regression",
    "Best Parameters": str(grid_search.best_params_),
    "CV Macro-F1": grid_search.best_score_,
    "Test Accuracy": accuracy,
    "Test Balanced Accuracy": balanced_acc,
    "Test Macro-F1": macro_f1,
    "Test Weighted-F1": weighted_f1,
}

log_reg_results_df = pd.DataFrame([log_reg_results])
log_reg_results_df.to_csv(results_dir / "logistic_regression_results.csv", index=False)
cv_results.to_csv(results_dir / "logistic_regression_cv_results.csv", index=False)

# Save confusion matrix figure
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(values_format="d", ax=ax)
ax.set_title("Logistic Regression Confusion Matrix")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(figures_dir / "logistic_regression_confusion_matrix.png", dpi=200)
plt.show()

log_reg_results_df.round(3)

## 12. Optional: compare with KNN result file

If your teammate saves a file called `results/knn_results.csv` with the same columns, this cell will combine both results.

In [ ]:
knn_path = Path("results/knn_results.csv")
log_path = Path("results/logistic_regression_results.csv")

if knn_path.exists():
    knn_results_df = pd.read_csv(knn_path)
    log_results_df = pd.read_csv(log_path)
    comparison_df = pd.concat([knn_results_df, log_results_df], ignore_index=True)
    comparison_df.to_csv(Path("results/model_comparison_knn_vs_logreg.csv"), index=False)
    display(comparison_df)
else:
    print("No results/knn_results.csv found yet.")
    print("Ask your teammate to save the KNN results with the same metric columns.")
    display(pd.read_csv(log_path))

## 13. Short interpretation for the presentation

Use this logic when explaining the comparison:

- Logistic Regression is a simple parametric model.
- It mainly learns linear decision boundaries.
- KNN is non-parametric and classifies based on nearby training samples.
- If KNN performs better, this suggests that local/non-linear class structure is useful for this dataset.
- Because the classes are imbalanced, macro-F1 is more meaningful than accuracy alone.